# NyayaAI - Training Notebook

End-to-end pipeline for **IndiaLex-FT** — a LoRA fine-tuned LLM for Indian legal and income-tax Q&A.

**Steps covered:**
1. Install dependencies
2. Load API keys from `.env`
3. Baseline evaluation (pre-training)
4. LoRA SFT fine-tuning
5. Post-training evaluation
6. LLM-as-Judge scoring via claude-sonnet-4-6
7. Download outputs

> **Note:** Run this notebook from the `indialex-ft/` directory. Ensure your `.env` file has `ANTHROPIC_API_KEY` and `GROQ_API_KEY` set before starting.

In [1]:
import os
from pathlib import Path

# Ensure the notebook working directory is indialex-ft/
notebook_dir = Path("__file__" if "__file__" in dir() else ".").resolve()
print(f"Working directory: {os.getcwd()}")

Working directory: /content


In [2]:
!pip install -r requirements.txt --quiet

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads indialex-ft/.env

def _check_secret(name, required=True):
    val = os.environ.get(name)
    if val:
        print(f"{name} loaded: True")
    else:
        status = "MISSING (required — add to .env)" if required else "skipped (optional)"
        print(f"{name} loaded: {status}")

_check_secret("ANTHROPIC_API_KEY", required=True)
_check_secret("GROQ_API_KEY",      required=True)
_check_secret("HF_TOKEN",          required=True)   # needed: Llama 3.2 is a gated model
_check_secret("WANDB_API_KEY",     required=False)

ANTHROPIC_API_KEY loaded: MISSING (required — add to .env)
GROQ_API_KEY loaded: MISSING (required — add to .env)
HF_TOKEN loaded: MISSING (required — add to .env)
WANDB_API_KEY loaded: skipped (optional)


In [4]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    raise EnvironmentError(
        "HF_TOKEN is not set. Llama 3.2 is a gated model — "
        "accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct "
        "and add HF_TOKEN=hf_... to your .env file."
    )

OSError: HF_TOKEN is not set. Llama 3.2 is a gated model — accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct and add HF_TOKEN=hf_... to your .env file.

## Step 1: Baseline Evaluation

Runs inference on the test split using the **base model** (before fine-tuning).
Computes ROUGE-L and BERTScore F1, and saves results to `evals/baseline_results.json`.

In [ ]:
!python scripts/baseline_eval.py

## Step 2: Training

Fine-tunes the base model with **LoRA + QLoRA (4-bit NF4)** via TRL SFTTrainer.
Saves the LoRA adapter to `outputs/adapter/` and the merged model to `outputs/merged/`.
Logs metrics to Weights & Biases.

In [ ]:
!python scripts/train.py

## Step 3: Post-training Evaluation

Runs inference on the test split using the **fine-tuned merged model**.
Computes ROUGE-L and BERTScore F1, compares against baseline, and saves
a delta summary to `evals/ft_results.json`.

In [ ]:
!python scripts/evaluate.py

## Step 4: LLM-as-Judge Evaluation

Uses **claude-sonnet-4-6** (Anthropic API) to score each fine-tuned answer on
four legal-domain dimensions (1–5 each): correctness, faithfulness, helpfulness,
and legal accuracy. Results saved to `evals/judge_results.json`.

In [ ]:
!python evals/llm_judge.py --results evals/ft_results.json

In [ ]:
import shutil
from pathlib import Path

for name, base_dir in [("nyayaai_outputs", "outputs"), ("nyayaai_evals", "evals")]:
    if Path(base_dir).exists():
        shutil.make_archive(name, "zip", root_dir=".", base_dir=base_dir)
        print(f"Zipped {base_dir}/ → {name}.zip")
    else:
        print(f"Skipped {base_dir}/ — directory does not exist yet.")

try:
    from google.colab import files
    for f in ["nyayaai_outputs.zip", "nyayaai_evals.zip"]:
        if Path(f).exists():
            files.download(f)
except ImportError:
    print("Zip files saved locally (not running in Colab).")